In [1]:
import sys
from pathlib import Path

project_root = Path(
    r"C:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\GlobalPPREstimation"
)
module_dir = str(project_root / "tools" / "scientific_code" / "PPREstimation")

if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

import pandas as pd
import numpy as np
from PPRCalculator import PPRCalculator
from ModelData import ModelData

# Loading

In [2]:
model_path = (
    "../models/"
    "27_118_Northwest_Africa_(1987)/"
    "model.json"
)

model = ModelData(
    model_path,
    model_name="Northwest Africa",
    model_year="1987",
)

# Inspect or edit these before constructing the calculator:
# model.groups_data['biomass_accum'] = np.nan
# model.groups_data
# model.DC
# model.det_fate

In [14]:
calculator = PPRCalculator.from_modeldata(
    modeldata=model,
    underdetermined=True,     # Estimate missing flows using LIM if True
    zero_catch=True,           # Replace missing catch with zero
    zero_biomass_accum=False,   # Replace missing biomass accumulation with zero
    default_gs=True,           # Supply GS=0.2 for regular groups missing GS
    weight_flow=1.0,           # LIM minimum-flow penalty weight
    weight_guess=1.0,          # LIM guess-deviation penalty weight
    normalize_DC=True,         # Normalize consumer diet sums if True
    DC_tol=0.001,             # Allowed deviation of diet sums from 1
    balance_BA_after_DC_normalization=True
)

C:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\GlobalPPREstimation\tools\scientific_code\PPREstimation\ModelData.py:617: RuntimeWarning: 2 group(s) route less than 100% of their flow_to_det to detritus; the remainder is treated as export out of the system: 5 (Sei whales), 4 (Brydes whales).
  warnings.warn(
C:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\GlobalPPREstimation\tools\scientific_code\PPREstimation\PPRCalculator.py:328: RuntimeWarning: Diet normalization compensated through runtime biomass accumulation for 14 Regular/PP group(s); computational closure, not observed biomass change. EE, M0 and detritus flows retained; see diet_normalization_balance_ledger.
  warnings.warn(


In [7]:
# calculator.get_det_fate()

# DC

In [14]:
model.DC.sum(axis=1).rename(index=model.seq2name).round(decimals=7)
# model.DC.rename(index=model.seq2name, columns=model.seq2name).round(decimals=7).columns
# model.DC.rename(index=model.seq2name, columns=model.seq2name).round(decimals=7)

diet_import               0.00000
Detritus                  0.00000
Phytoplankton             0.00000
Zooplankton               1.00000
Benthic producers         0.00000
Benthos                   0.99990
Crustaceans               1.00000
Cephalopods               1.00000
Other coastal pelagics    1.00000
Clupeids                  1.00000
Coastal demersal          1.00002
Coastal tunas             1.00002
Rays                      1.00000
Sharks                    1.00000
Bathydeersal predators    1.00000
Mesopelagic predators     1.00000
Large pelagics            1.00002
Seabirds                  1.00000
Dolphins                  1.00000
Killer whales             1.00000
Sperm whales              0.99970
Beaked whales             1.00000
Baleen whales             1.00000
Sei whales                1.00010
Brydes whales             1.00000
Humpback whales           1.00000
Fin whales                0.99960
Minke whales              1.00000
dtype: float64

In [7]:
calculator.get_DC().sum(axis=1).round(decimals=3)

39    0.0
38    0.0
37    1.0
36    1.0
35    1.0
34    1.0
33    1.0
32    1.0
31    1.0
30    1.0
29    1.0
28    1.0
27    1.0
26    1.0
25    1.0
24    1.0
23    1.0
22    1.0
21    1.0
20    1.0
19    1.0
18    1.0
17    1.0
16    1.0
15    1.0
14    1.0
13    1.0
12    1.0
11    1.0
10    1.0
9     1.0
8     1.0
7     1.0
6     1.0
5     1.0
4     1.0
3     1.0
2     0.0
1     0.0
dtype: float64

# Groups Data

In [17]:
# model.groups_data.round(decimals=7)[['group_name', 'ge', 'ee']]
model.groups_data.round(decimals=7)[['group_name', 'gs', 'biomass_accum']]

,group_name,gs,biomass_accum
group_seq,,,
27,Detritus,0.0,5934.798
26,Phytoplankton,0.0,0.000
25,Zooplankton,0.2,0.000
24,Benthic producers,0.0,0.000
23,Benthos,0.2,0.000
22,Crustaceans,0.2,0.000
21,Cephalopods,0.2,0.000
20,Other coastal pelagics,0.2,0.000
19,Clupeids,0.2,0.000


In [5]:
# calculator.get_groups_df().round(decimals=7)[['group_name', 'ge', 'ee', 'tl']]
calculator.get_groups_df().round(decimals=7)[['group_name', 'gs', 'biomass_accum']]

,group_name,gs,biomass_accum
group_seq,,,
28,diet_import,0.0,0.000000e+00
27,Detritus,0.0,5.934459e+03
26,Phytoplankton,0.0,1.112800e-03
25,Zooplankton,0.2,1.107400e-03
24,Benthic producers,0.0,-1.863520e-02
23,Benthos,0.2,-7.563400e-03
22,Crustaceans,0.2,-2.000000e-07
21,Cephalopods,0.2,1.570000e-05
20,Other coastal pelagics,0.2,1.380000e-04


# Is Model Balanced

In [6]:
balanced, p, q = calculator.is_model_balanced()
balanced

True

In [10]:
(100 * (p - calculator.p)/p).rename(index=calculator.seq2name)

group_seq
diet_import               0.000000e+00
Detritus                  0.000000e+00
Phytoplankton            -1.502198e-05
Zooplankton              -9.630345e-05
Benthic producers         1.651867e-03
Benthos                   1.547747e-03
Crustaceans               2.850862e-07
Cephalopods              -1.482297e-04
Other coastal pelagics   -4.000549e-04
Clupeids                 -2.496509e-04
Coastal demersal          1.361736e-07
Coastal tunas             1.903411e-08
Rays                     -4.606569e-05
Sharks                   -1.328985e-05
Bathydeersal predators    8.918273e-07
Mesopelagic predators     1.549141e-06
Large pelagics           -5.099728e-06
Seabirds                  0.000000e+00
Dolphins                  2.074468e-06
Killer whales             0.000000e+00
Sperm whales              0.000000e+00
Beaked whales             0.000000e+00
Baleen whales             0.000000e+00
Sei whales                0.000000e+00
Brydes whales             0.000000e+00
Humpback whales

In [11]:
(100 * (q - calculator.q)/q).rename(index=calculator.seq2name)

group_seq
diet_import               0.000000e+00
Detritus                  0.000000e+00
Phytoplankton            -1.502198e-05
Zooplankton              -2.147320e-05
Benthic producers         1.651867e-03
Benthos                   2.049504e-04
Crustaceans              -1.189376e-06
Cephalopods              -4.063792e-05
Other coastal pelagics   -9.538722e-05
Clupeids                 -4.911334e-05
Coastal demersal          3.679078e-06
Coastal tunas             4.135657e-09
Rays                      6.048407e-08
Sharks                   -2.371858e-06
Bathydeersal predators   -5.428404e-06
Mesopelagic predators     2.137794e-07
Large pelagics           -4.197330e-06
Seabirds                  0.000000e+00
Dolphins                  7.096589e-09
Killer whales             2.458197e-06
Sperm whales              3.940614e-06
Beaked whales             0.000000e+00
Baleen whales             2.772467e-06
Sei whales               -1.755054e-14
Brydes whales             0.000000e+00
Humpback whales

In [8]:
calc_balanced = calculator.balanced_model
print(calc_balanced.n_balance_runs)
balanced, p, q = calc_balanced.is_model_balanced()
100 * (p - calc_balanced.p)/p

1


group_seq
25             NaN
24    0.000000e+00
23    0.000000e+00
22    0.000000e+00
21    0.000000e+00
20    0.000000e+00
19    0.000000e+00
18    0.000000e+00
17    0.000000e+00
16    0.000000e+00
15    0.000000e+00
14   -1.731950e-14
13    0.000000e+00
12    0.000000e+00
11    0.000000e+00
10    0.000000e+00
9    -2.107821e-14
8     0.000000e+00
7     0.000000e+00
6     0.000000e+00
5     0.000000e+00
4    -2.132190e-14
3     0.000000e+00
2     0.000000e+00
1     0.000000e+00
dtype: float64

# Diagnostics

In [21]:
result, sppr, A, L = calculator.diagnose_sppr(
    TE_option='GE',
    return_sppr=True
)

In [26]:
result, sppr, A, L = calculator.diagnose_sppr(
    TE_option='TE',
    return_sppr=True
)

C:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\GlobalPPREstimation\tools\scientific_code\PPREstimation\PPRCalculator.py:2706: RuntimeWarning: 10 group(s) have TE=0 because EE=0 (M0=p): 11 (Seabirds), 9 (Killer whales), 8 (Sperm whales), 7 (Beaked whales), 6 (Baleen whales), 5 (Sei whales), 4 (Brydes whales), 3 (Humpback whales), 2 (Fin whales), 1 (Minke whales). Their consumed PP is re-credited to detritus (fix_EE_0_cases=True) so the TE SPPR stays PP-balanced.
  warnings.warn(
C:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\GlobalPPREstimation\tools\scientific_code\PPREstimation\PPRCalculator.py:2731: RuntimeWarning: 1 group(s) have near-zero TE (0 < EE << 1): 17 (Coastal tunas). Their SPPR is near-singular (inherent to the TE method); the detritus SPPR may be inflated and is not corrected by fix_EE_0_cases.
  warnings.warn(


In [22]:
result['model_input']

{'status': 'WARN',
 'is_model_balanced': True,
 'p_max_rel_residual': 9.151465484541914e-08,
 'q_max_rel_residual': 8.051499898175876e-08,
 'dc_rows_sum_to_1': True,
 'dc_max_deviation': 2.220446049250313e-16,
 'n_negative_catch': 0,
 'n_zero_catch': 16,
 'total_catch': 0.5164791377,
 'has_catch': True,
 'has_ee_issues': True,
 'n_ee0': 10,
 'n_ee_marginal': 0,
 'n_ee_gt_1': 0,
 'ee0_groups': [11, 9, 8, 7, 6, 5, 4, 3, 2, 1],
 'ee_marginal_groups': []}

In [27]:
result['divergence']

{'status': 'WARN',
 'solve_error': None,
 'b': 0.0,
 'b_converges': True,
 'rho_living': 0.9907169786465047,
 'living_converges': True,
 'sppr_det': {27: 1.6080319213502223},
 'max_sppr_det': 1.6080319213502223,
 'max_sppr_group': {'seq': 13,
  'tl': 3.9298839512443298,
  'sppr': 2774448.0998087376,
  'inv_te': 61.919811165406536},
 'max_tl_group': {'seq': 7,
  'tl': 4.199027795682816,
  'sppr': 0.0,
  'inv_te': None},
 'n_negative_sources': 0,
 'expect_negatives': False,
 'near_singular_te': [17]}

In [24]:
result['balance']

{'status': 'OK',
 'is_balanced': False,
 'inflow': 12460.853469902122,
 'outflow': 12459.671285297702,
 'rel_gap': 9.487188074844486e-05}

In [25]:
result['footprint']

{'ppr_all': 12.299820660231267,
 'ppr_inner': 12.299820660231267,
 'ppr_pp_only': 9.862088187769473,
 'npp': 12460.712646,
 'ppr2npp': 0.0009870880590589352,
 'ppr2npp_pp_only': 0.0007914545875460254}

In [28]:
result['config']

{'TE_option': 'GE',
 'det_open_mode': 'none',
 'det_theta': 1.0,
 'det_external_sppr': 0.0,
 'det_collapse_mode': 'never',
 'explicit_TE': False,
 'method': 'single_detritus',
 'would_pool': False,
 'model': 'East China Sea (2018)'}

In [29]:
result['warnings']

['total catch is 0: every footprint number is identically 0 (the divergence diagnostics are unaffected)',
 "2 group(s) with EE=0 (all production is non-predatory death): 23 (Marine mammals), 22 (Sharks). Under TE_option='TE' their TE row is 0, which severs them from the nullspace and leaks the PP they consumed"]

In [14]:
(sppr<0).any().any()

np.False_